In [ ]:
import os
import numpy as np

from evaluate import load

from BudaOCR.Encoder import LabelEncoder, WylieEncoder
from BudaOCR.Networks import Easter2PlusNetwork
from BudaOCR.Trainer import OCRTrainer

from BudaOCR.Config import CHARSET
from BudaOCR.Utils import (
    build_data_paths,
    create_dir,
    shuffle_data,
    show_image
    )

In [ ]:
# local dir
#dataset_path = "D:/Datasets/KhyentseWangpo"
dataset_path = "E:/Datasets/rNam-rgyal_OCR-Dataset/Dataset"
image_paths, label_paths = build_data_paths(dataset_path, img_file_ext="jpg")
image_paths, label_paths = shuffle_data(image_paths, label_paths)

print(f"Images: {len(image_paths)}, Labels: {len(label_paths)}")

output_dir = os.path.join("Output")
create_dir(output_dir)

In [ ]:
image_width = 3200
image_height = 100
encoder = WylieEncoder(CHARSET)
num_classes = encoder.num_classes()

In [ ]:
network = Easter2PlusNetwork(image_width, image_height, num_classes=num_classes, easter_variant="fixed")
batch_size = 32
workers = 4

In [ ]:
ocr_trainer = OCRTrainer(
    network=network,
    label_encoder=encoder,
    workers=workers, 
    image_width=image_width,
    image_height=image_height,
    batch_size=batch_size, 
    output_dir=output_dir, 
    preload_labels=True
    )

ocr_trainer.init(image_paths, label_paths)

In [ ]:
num_epochs = 36
ocr_trainer.train(epochs=num_epochs, check_cer=True, export_onnx=True, silent=False)

In [ ]:
# check dataloader
test_sample = next(iter(ocr_trainer.test_loader))

test_img = test_sample[0][4].numpy()
test_img = np.transpose(test_img, axes=[1, 2, 0])

show_image(test_img)

In [ ]:
# debug start logits for non-zeros
test_sample = next(iter(ocr_trainer.test_loader))
test_logits, gt_labels = ocr_trainer.network.test(test_sample)
pred = np.argmax(test_logits[0], axis=0)
print(pred[:40]) # there should be no leading zeros